### Pré-processamento do Benchmark Dataset (Tsinghua Lab) para o participante S1

Este notebook organiza os dados do participante S1 no formato esperado pelo MNE, cria os objetos Epochs, aplica os filtros temporais e salva os resultados para o notebook de visualização.

In [1]:
# Carregar os dados

import numpy as np
import pandas as pd

data = np.load('data/output/S1_pre_pros.npy')

print(data.dtype)
print(data.shape)

channels = pd.read_csv('data/input/64-channels.loc', sep=r'\s+', header=None, usecols=[3])[3].tolist() # Lê 64 canais do arquivo '64-channels.loc'
fs = 250
electrode_index, time_points, target_index, block_index = data.shape

float64
(64, 1500, 40, 6)


**Reorganização do vetor**

Para isso, cada par (bloco, alvo) virou uma *epoch* de formato [64, 1500], sendo 6 blocos x 40 alvos (e trials) = 240 epochs. 

In [2]:
# Reordena o array para [bloco, alvo, canal, tempo] e junta bloco e alvo em um único índice.
import mne

epochs_all = data.transpose(3, 2, 0, 1)
print(epochs_all.shape)

epochs_all = epochs_all.reshape(block_index * target_index, electrode_index, time_points)
print(epochs_all.shape)

(6, 40, 64, 1500)
(240, 64, 1500)


**Verificação**

Como cada época deve ser idêntica ao trecho correspondente do array original, o trecho abaixo compara a época 0 (bloco 0, alvo 0), a época 21 (bloco 1, alvo 1) e a época 239 (bloco 5, alvo 39). Como o *reshape* percorre os blocos e, dentro de cada bloco, os alvos, a *epoch* de índice `b * 40 + t` correponde a `(bloco b, alvo t)`.

In [3]:
target_labels = np.tile(np.arange(target_index), block_index)

print(np.array_equal(epochs_all[0], data[:, :, 0, 0]))
print(np.array_equal(epochs_all[41], data[:, :, 1, 1]))
print(np.array_equal(epochs_all[239], data[:, :, 39, 5]))
print(target_labels[0:44])
print(target_labels.shape)

True
True
True
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39  0  1  2  3]
(240,)


**Objeto info**

O info reúne a frequência de amostragem de 250 Hz e os nomes dos 64 canais lidos do arquivo de posições, todos declarados como canais de EEG.

In [4]:
sfreq = fs
ch_names = channels

info = mne.create_info(ch_names, sfreq=sfreq, ch_types='eeg')
info

<Info | 7 non-empty values
 bads: []
 ch_names: FP1, FPZ, FP2, AF3, AF4, F7, F5, F3, F1, FZ, F2, F4, F6, F8, ...
 chs: 64 EEG
 custom_ref_applied: False
 highpass: 0.0 Hz
 lowpass: 125.0 Hz
 meas_date: unspecified
 nchan: 64
 projs: []
 sfreq: 250.0 Hz
>

**Criação do EpochsArray**

Com o info pronto, foram criados os eventos, que dizem ao MNE qual alvo foi olhado em cada epoch. Como o MNE assume que o sinal está em volts e os dados estão em microvolts, estes foram multiplicados por 1 × 10⁻⁶ ao criar o objeto. 

O tmin foi definido como -0,5 s porque cada epoch começa 500 ms antes do estímulo, de modo que o zero do eixo de tempo marca o início do estímulo. O resultado é um único objeto com as 240 epochs, cada uma com 1500 pontos, indo de -0,5 s até cerca de 5,5 s.

In [5]:
from scipy.io import loadmat

# Frequência de cada um dos 40 alvos
freq_phase = loadmat('data/input/Freq_Phase.mat')
print(freq_phase.keys())

freqs = freq_phase['freqs'].flatten()
print(freqs)

dict_keys(['__header__', '__version__', '__globals__', 'phases', 'freqs'])
[ 8.   9.  10.  11.  12.  13.  14.  15.   8.2  9.2 10.2 11.2 12.2 13.2
 14.2 15.2  8.4  9.4 10.4 11.4 12.4 13.4 14.4 15.4  8.6  9.6 10.6 11.6
 12.6 13.6 14.6 15.6  8.8  9.8 10.8 11.8 12.8 13.8 14.8 15.8]


In [6]:
pre_stimulus_time = 0.5

# Eventos: índice da epoch, valor anterior e código do alvo
events = np.column_stack((
    np.arange(len(target_labels)),
    np.zeros(len(target_labels), dtype=int),
    target_labels
))

# Nome de cada código de alvo, dado pela sua frequência
event_id = {}
for target in range(target_index):
    event_id[str(round(freqs[target], 1))] = target

# Bloco e alvo de cada epoch, para selecionar por bloco nas visualizações
metadata = pd.DataFrame({
    'block': np.repeat(np.arange(block_index), target_index),
    'target': target_labels
})

# Objeto Epochs em volts, com tempo zero no início do estímulo
mne_epochs = mne.EpochsArray(epochs_all * 1e-6, info, events, tmin=-pre_stimulus_time, event_id=event_id, metadata=metadata)

print(mne_epochs)

Adding metadata with 2 columns
240 matching events found
No baseline correction applied
0 projection items activated
<EpochsArray | 240 events (all good), -0.5 – 5.496 s (baseline off), ~175.8 MiB, data loaded, with metadata,
 '8.0': 6
 '9.0': 6
 '10.0': 6
 '11.0': 6
 '12.0': 6
 '13.0': 6
 '14.0': 6
 '15.0': 6
 '8.2': 6
 '9.2': 6
 and 30 more events ...>


**Aplicação de filtros**

Para filtrar, foi utilizado o método filter do próprio objeto Epochs do MNE porque ele já aplica o filtro em cada canal de todas as epochs, sem deslocar a fase do sinal.

Como as frequências dos alvos vão de 8 a 15,8 Hz e o segundo harmônico chega a 31,6 Hz, foi aplicado um passa-faixa de 6 a 35 Hz, que mantém as frequências fundamentais e os segundos harmônicos e remove a oscilações de baixa frequência e o ruído de frequências altas. O ruído da touca de 50 Hz já fica fora dessa faixa, e o filtro notch também já havia sido aplicado na gravação.

O filtro foi aplicado em uma cópia, para manter o objeto original sem filtro e poder comparar os dois no domínio da frequência. 

In [7]:
# Passa-faixa de 6 a 35 Hz em uma cópia do objeto
mne_filtered = mne_epochs.copy().filter(l_freq=6., h_freq=35.)

# Salva os dois objetos para o notebook de visualização
mne_epochs.save('data/output/S1_epo.fif', overwrite=True)
mne_filtered.save('data/output/S1_filtered_epo.fif', overwrite=True)

Setting up band-pass filter from 6 - 35 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 6.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 5.00 Hz)
- Upper passband edge: 35.00 Hz
- Upper transition bandwidth: 8.75 Hz (-6 dB cutoff frequency: 39.38 Hz)
- Filter length: 413 samples (1.652 s)

Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.
Overwriting existing file.


[PosixPath('/home/duda/Faculdade/OPT098/fbcca/data/output/S1_filtered_epo.fif')]